In [4]:
import sys

!{sys.executable} -m pip install -U langchain langchain-community langchain-text-splitters
!{sys.executable} -m pip install -U langchain-huggingface langchain-pinecone
!{sys.executable} -m pip install -U "pinecone<8.0.0" pypdf sentence-transformers

In [5]:
from langchain_community.document_loaders import PyPDFLoader

In [6]:
load = PyPDFLoader("/content/AI-Engineer-Interview-Questions-PDF.pdf")

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

doc = load.load()

split = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = split.split_documents(doc)

print(len(chunks))

366


In [9]:
from langchain_huggingface import HuggingFaceEmbeddings

In [10]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
import os
import time
import getpass

In [14]:
from getpass import getpass
from pinecone import Pinecone

api_key = getpass("Enter your Pinecone API Key: ")
pc = Pinecone(api_key=api_key)
#pcsk_4noqy_GzNyfrWWvpJ3DGqdYUUKY5at5g3uX7VW9FcjGBcskTCeZ4N69EdkrE2ssAZLFQS
print("Pinecone client initialized successfully!")

Enter Pinecone API Key: ··········


In [15]:
index_name = "rag-pdf-project"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

In [17]:
index = pc.Index(index_name)
DB = PineconeVectorStore(
    index=index,
    embedding=embeddings
)

DB.add_documents(chunks)
print("Documents stored successfully in Pinecone DB!")

question = "What are the AI safety considerations when deploying LLM applications?"
result = DB.similarity_search(question, k=3)

for i in result:
    print(i.page_content)
    print("-" * 50)

Documents stored successfully in Pinecone DB!
production depth signals. Probe: "Your LLM's structured output fails about 2% of the time despite 
JSON mode. How do you handle this in production?" 
Q85. What are the AI safety considerations when deploying LLM applications? 
What a Strong Answer Covers: Jailbreaking; data privacy; output toxicity; bias; overreliance; 
legal/compliance; responsible disclosure. 
Strong Answer: "LLM safety in production spans several dimensions: (1) Jailbreaking and misuse -
--------------------------------------------------
LLM outputs against expected schemas before execution, especially for agentic systems. (4) 
Sandboxed execution - any code or commands the LLM generates should run in a sandboxed 
environment. (5) Monitoring - log and audit all LLM interactions for anomalies. Prompt injection is 
the #1 security concern for LLM applications in production." 
Weak Answer: "Prompt injection is a security attack where users try to manipulate the model. 
Sani